# Mi primera red convolucional: perritos vs. gatitos

_Notebook creado por Zaid De Anda_

En este ejercicio construiremos una red neuronal convolucional (CNN) sencilla que responde una sola pregunta:

> ¿La imagen contiene un gato o un perro?

El objetivo no es crear el mejor modelo posible, sino entender el flujo completo:

1. cargar imágenes,
2. ver cómo las recibe la computadora,
3. construir una CNN pequeña,
4. entrenarla,
5. revisar sus predicciones.

## 1. Preparación

Usaremos **PyTorch** y las imágenes de gatos y perros incluidas en **CIFAR-10**. Descargaremos el dataset desde el CDN de **Hugging Face**, que suele ser más rápido que el servidor original, y conservaremos únicamente estas dos clases.

In [1]:
import importlib.util
import subprocess
import sys

for package in ["torch", "torchvision", "matplotlib", "datasets"]:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Dispositivo:", device)

## 2. Descargar y cargar las imágenes

CIFAR-10 contiene imágenes pequeñas de `32 × 32` píxeles. El dataset tiene diez categorías, pero seleccionaremos solamente gatos y perros. Después transformaremos las imágenes en tensores: arreglos de números que PyTorch puede procesar.

In [ ]:
image_transform = transforms.Compose([
    transforms.ToTensor(),
])

# Esta versión se descarga desde el CDN de Hugging Face.
complete_train_dataset = load_dataset("uoft-cs/cifar10", split="train")
complete_validation_dataset = load_dataset("uoft-cs/cifar10", split="test")

# En CIFAR-10, la etiqueta 3 es gato y la etiqueta 5 es perro.
wanted_labels = {3: 0, 5: 1}
complete_train_dataset = complete_train_dataset.filter(
    lambda example: example["label"] in wanted_labels
)
complete_validation_dataset = complete_validation_dataset.filter(
    lambda example: example["label"] in wanted_labels
)

class CatsDogsDataset(Dataset):
    def __init__(self, hf_dataset, transform):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        example = self.dataset[index]
        image = self.transform(example["img"])
        label = wanted_labels[example["label"]]
        return image, label


train_dataset = CatsDogsDataset(complete_train_dataset, image_transform)
validation_dataset = CatsDogsDataset(complete_validation_dataset, image_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False, num_workers=0)

class_names = ["cat", "dog"]
print("Clases:", class_names)
print("Imágenes de entrenamiento:", len(train_dataset))
print("Imágenes de validación:", len(validation_dataset))

Las etiquetas se asignan automáticamente según el nombre de las carpetas:

- `0` → cats
- `1` → dogs

In [ ]:
images, labels = next(iter(train_loader))

plt.figure(figsize=(10, 6))
for index in range(12):
    plt.subplot(3, 4, index + 1)
    plt.imshow(images[index].permute(1, 2, 0))
    plt.title(class_names[labels[index].item()])
    plt.axis("off")

plt.tight_layout()
plt.show()

print("Forma de un lote:", images.shape)
print("32 imágenes, 3 canales de color, 32 píxeles de alto y 32 de ancho")

## 3. Una CNN pequeña

Nuestra red tiene dos bloques convolucionales:

- **Convolución:** aprende filtros para reconocer patrones como bordes y texturas.
- **ReLU:** conserva las señales útiles.
- **MaxPool:** reduce el tamaño de la imagen y se queda con lo más importante.

Al final, una capa lineal decide entre dos resultados: gato o perro.

In [ ]:
class CatsDogsCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),       # 32 x 32 -> 16 x 16
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),       # 16 x 16 -> 8 x 8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = CatsDogsCNN().to(device)
print(model)

trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parámetros entrenables: {trainable_parameters:,}")

## 4. Entrenamiento

Durante cada época la red ve todas las imágenes de entrenamiento una vez. Para cada lote:

1. hace una predicción,
2. calcula qué tan equivocada estuvo (`loss`),
3. ajusta sus parámetros.

Usaremos pocas épocas para mantener el ejercicio rápido.

In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
EPOCHS = 5

history = {"train_loss": [], "train_accuracy": [], "validation_accuracy": []}

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / len(train_loader)
    train_accuracy = correct / total

    model.eval()
    validation_correct = 0
    validation_total = 0

    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            predictions = model(images).argmax(dim=1)
            validation_correct += (predictions == labels).sum().item()
            validation_total += labels.size(0)

    validation_accuracy = validation_correct / validation_total
    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["validation_accuracy"].append(validation_accuracy)

    print(
        f"Época {epoch + 1}/{EPOCHS} | "
        f"loss: {train_loss:.3f} | "
        f"accuracy entrenamiento: {train_accuracy:.3f} | "
        f"accuracy validación: {validation_accuracy:.3f}"
    )

## 5. ¿Aprendió?

`Accuracy` es la proporción de predicciones correctas. Comparamos entrenamiento y validación porque nos interesa que el modelo funcione con imágenes que no utilizó para aprender.

In [ ]:
epochs = range(1, EPOCHS + 1)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], marker="o")
plt.title("Pérdida de entrenamiento")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(epochs, history["train_accuracy"], marker="o", label="Entrenamiento")
plt.plot(epochs, history["validation_accuracy"], marker="o", label="Validación")
plt.title("Accuracy")
plt.xlabel("Época")
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Veamos algunas predicciones

El color del título indica si la predicción fue correcta. Verde significa acierto y rojo significa error.

In [ ]:
model.eval()
images, labels = next(iter(validation_loader))

with torch.no_grad():
    outputs = model(images.to(device))
    probabilities = torch.softmax(outputs, dim=1)
    predictions = probabilities.argmax(dim=1).cpu()
    confidence = probabilities.max(dim=1).values.cpu()

plt.figure(figsize=(12, 8))
for index in range(12):
    prediction = predictions[index].item()
    real_label = labels[index].item()
    color = "green" if prediction == real_label else "red"

    plt.subplot(3, 4, index + 1)
    plt.imshow(images[index].permute(1, 2, 0))
    plt.title(
        f"Predicción: {class_names[prediction]}\n"
        f"Confianza: {confidence[index]:.0%}",
        color=color,
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

## 7. Detectar gatos y perros en una imagen con YOLO

Nuestra CNN anterior **clasifica la imagen completa** como gato o perro. YOLO hace algo distinto: es un modelo de **detección de objetos**, por lo que puede encontrar varios animales y dibujar una caja alrededor de cada uno.

Usaremos un YOLO pequeño ya entrenado con el dataset COCO. No lo entrenaremos de nuevo; solamente haremos **inferencia**.

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])

from ultralytics import YOLO

# La primera ejecución descargará automáticamente los pesos del modelo.
yolo_model = YOLO("yolov8n.pt")

### Sube tu imagen

En Google Colab aparecerá un botón para seleccionar el archivo. Si estás usando Jupyter en tu computadora, escribe la ruta de una imagen local cuando la celda lo solicite.

In [ ]:
from pathlib import Path

try:
    from google.colab import files

    uploaded_files = files.upload()
    if not uploaded_files:
        raise ValueError("No se seleccionó ninguna imagen.")
    image_path = Path(next(iter(uploaded_files)))
except ImportError:
    image_path = Path(input("Ruta de la imagen: ").strip()).expanduser()

if not image_path.is_file():
    raise FileNotFoundError(f"No se encontró la imagen: {image_path}")

print("Imagen seleccionada:", image_path)

In [ ]:
# En el dataset COCO: 15 = cat y 16 = dog.
#Si quieres ver todas las clases, revisa https://gist.github.com/rcland12/dc48e1963268ff98c8b2c4543e7a9be8
yolo_results = yolo_model.predict(
    source=str(image_path),
    classes=[15, 16], 
    conf=0.25,
    verbose=False,
)

result = yolo_results[0]
annotated_image = result.plot()[:, :, ::-1]  # BGR -> RGB

plt.figure(figsize=(10, 8))
plt.imshow(annotated_image)
plt.axis("off")
plt.title("Detecciones de YOLO")
plt.show()

if len(result.boxes) == 0:
    print("YOLO no encontró gatos ni perros con suficiente confianza.")
else:
    print(f"Objetos detectados: {len(result.boxes)}")
    for box in result.boxes:
        class_id = int(box.cls.item())
        confidence = box.conf.item()
        print(f"- {result.names[class_id]}: {confidence:.1%} de confianza")

## 8. Conclusiones

En este notebook:

- cargamos imágenes desde carpetas,
- transformamos cada imagen en un tensor,
- construimos una CNN con dos convoluciones,
- entrenamos un clasificador de gatos y perros,
- evaluamos el modelo con datos que no había visto,
- usamos un YOLO preentrenado para localizar gatos y perros en una imagen propia.

### Preguntas para discutir

1. ¿Qué errores comete el modelo?
2. ¿La confianza siempre significa que la predicción es correcta?
3. ¿Qué pasaría si entrenamos durante más épocas?
4. ¿Cómo cambiaría el resultado si usamos imágenes más grandes?

### Reto opcional

Cambia `EPOCHS` de 5 a 10 o agrega una tercera convolución. ¿Mejora la precisión de validación?